In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

In [2]:
# 1. USER IDS

student_ids = [f"STU{str(i).zfill(5)}" for i in range(1, 21)]
faculty_ids = [f"FAC{str(i).zfill(3)}" for i in range(1, 9)]
staff_ids = [f"OFF{str(i).zfill(3)}" for i in range(1, 7)]
admin_ids = ["ADMIN001"]

In [3]:
# 2. WINGS + VOCAB

wings_map = {
    "Plumbing": ["leak", "tap", "flush", "geyser", "drain", "overflow", "pipe"],
    "Electrical": ["fan", "light", "socket", "MCB", "switchboard", "sparking", "wiring"],
    "IT": ["wifi", "internet", "portal", "login", "router", "network"],
    "Carpentry": ["door", "lock", "hinge", "cupboard", "window", "drawer"],
    "HVAC": ["AC", "cooling", "ventilation", "filter", "airflow"],
    "Cleaning": ["garbage", "dustbin", "mopping", "odour", "trash", "cleanliness"]
}

In [4]:
# 3. ROLE-BASED WING

def role_based_wing(user_id):
    if user_id.startswith("STU"):
        return random.choices(["Electrical", "Plumbing", "Cleaning"], [0.4, 0.35, 0.25])[0]
    elif user_id.startswith("FAC"):
        return random.choices(["IT", "HVAC", "Electrical"], [0.5, 0.3, 0.2])[0]
    elif user_id.startswith("OFF"):
        return random.choices(["Cleaning", "Carpentry", "Electrical"], [0.5, 0.3, 0.2])[0]
    else:
        return random.choice(list(wings_map.keys()))

In [5]:
# 4. BALANCED WING DISTRIBUTION

def generate_balanced_wings(n):
    wings = list(wings_map.keys())
    min_per_wing = int(0.12 * n)

    wing_list = []

    # Ensure minimum count per wing
    for w in wings:
        wing_list += [w] * min_per_wing

    # Fill remaining with weighted realism
    remaining = n - len(wing_list)

    weighted_fill = random.choices(
        ["Electrical", "Plumbing", "Cleaning", "Carpentry", "IT", "HVAC"],
        weights=[0.25, 0.22, 0.18, 0.15, 0.12, 0.08],
        k=remaining
    )

    wing_list += weighted_fill
    random.shuffle(wing_list)

    return wing_list

In [6]:
# 5. TEXT GENERATION

openers = [
    "I am facing an issue where",
    "There seems to be a problem with",
    "I noticed that",
    "Currently experiencing an issue with",
    "There is something wrong with"
]

connectors = ["and also", "along with that", "additionally", "on top of that"]

contexts = [
    "in my hostel room",
    "in the lab",
    "near the washroom",
    "in the corridor",
    "in the common area"
]

urgency_phrases = [
    "This needs immediate attention.",
    "Please resolve this urgently.",
    "This is causing inconvenience.",
    "Kindly look into this as soon as possible."
]

real_sentences = [
    "fan not working and making noise",
    "wifi disconnecting frequently",
    "water leaking continuously from pipe",
    "garbage not collected for days",
    "door lock broken and unsafe"
]

In [7]:
def generate_description(primary_wing):
    item1 = random.choice(wings_map[primary_wing])
    sentence = f"{random.choice(openers)} the {item1} is not functioning properly"

    if random.random() < 0.7:
        sentence += f" {random.choice(contexts)}"

    # Multi-label noise
    if random.random() < 0.3:
        other_wing = random.choice(list(wings_map.keys()))
        if other_wing != primary_wing:
            item2 = random.choice(wings_map[other_wing])
            sentence += f", {random.choice(connectors)} the {item2} is also not working"

    if random.random() < 0.6:
        sentence += ". " + random.choice(urgency_phrases)

    if random.random() < 0.15:
        sentence = random.choice(real_sentences)

    if random.random() < 0.1:
        sentence = sentence.lower()

    return sentence

In [8]:
# 6. LOCATION GENERATION

academic_blocks = list(range(1, 8))
residencies = ["A", "B", "C"]
pseudo_areas = ["CSE", "LS", "ES", "H&L", "M", "XYZ"]

def _academic_room_code():
    area = random.choice(pseudo_areas)
    room_no = random.randint(101, 799)
    return f"{area}-{room_no}"

def _residency_room_code():
    room_no = random.randint(101, 399)
    part = random.choice(["A", "B"])
    return f"{room_no}-{part}"

def generate_location(user_id):
    # Students are mostly from residencies; occasionally from academic blocks.
    if user_id.startswith("STU"):
        if random.random() < 0.75:
            res = random.choice(["A", "B"])
            return f"Residencies ({res}) : Room No ({_residency_room_code()})"
        block = random.choice(academic_blocks)
        return f"Academic Block ({block}) : Room No ({_academic_room_code()})"

    # Faculty can be from C residency or academic blocks.
    if user_id.startswith("FAC"):
        if random.random() < 0.35:
            return f"Residencies (C) : Room No ({_residency_room_code()})"
        block = random.choice(academic_blocks)
        return f"Academic Block ({block}) : Room No ({_academic_room_code()})"

    # Staff/Admin are mostly tied to academic spaces.
    block = random.choice(academic_blocks)
    return f"Academic Block ({block}) : Room No ({_academic_room_code()})"

In [9]:
# 7. PRIORITY + STATUS

high_priority_words = ["urgent", "fire", "sparking", "overflow", "short circuit"]

def assign_priority(description):
    text = description.lower()
    if any(word in text for word in high_priority_words):
        return "High"
    return random.choices(["Medium", "Low"], [0.7, 0.3])[0]

def assign_status(timestamp):
    age = (datetime.now() - timestamp).days
    if age > 10:
        return random.choices(["Resolved", "In Progress"], [0.8, 0.2])[0]
    elif age > 3:
        return random.choices(["In Progress", "Pending"], [0.6, 0.4])[0]
    return "Pending"

In [10]:
# 8. LABEL NOISE

def inject_label_noise(true_wing, prob=0.08):
    if random.random() < prob:
        return random.choice([w for w in wings_map if w != true_wing])
    return true_wing

In [11]:
# 9. USER DISTRIBUTION

def generate_user_sequence(n):
    n_students = int(0.65 * n)
    n_faculty = int(0.25 * n)
    n_staff = int(0.09 * n)
    n_admin = n - (n_students + n_faculty + n_staff)

    users = []
    users += random.choices(student_ids, k=n_students)
    users += random.choices(faculty_ids, k=n_faculty)
    users += random.choices(staff_ids, k=n_staff)
    users += admin_ids * n_admin

    random.shuffle(users)
    return users

In [12]:
# 10. MAIN GENERATOR

def generate_dataset(n=600):
    data = []
    start_time = datetime.now() - timedelta(days=30)

    user_sequence = generate_user_sequence(n)
    wing_sequence = generate_balanced_wings(n)

    for i in range(n):
        user_id = user_sequence[i]

        # Blend realism + balance
        if random.random() < 0.7:
            true_wing = wing_sequence[i]
        else:
            true_wing = role_based_wing(user_id)

        description = generate_description(true_wing)
        location = generate_location(user_id)

        timestamp = start_time + timedelta(
            minutes=random.randint(0, 30 * 24 * 60)
        )

        data.append({
            "ticket_id": f"TICK_{1000+i}",
            "user_id": user_id,
            "timestamp": timestamp.strftime("%Y-%m-%d %H:%M"),
            "description": description,
            "location": location,
            "wing": inject_label_noise(true_wing),
            "priority": assign_priority(description),
            "status": assign_status(timestamp),
            "true_wing": true_wing
        })

    df = pd.DataFrame(data)
    df = df.sample(frac=1).reset_index(drop=True)

    return df

In [13]:
# 11. EXECUTE

df = generate_dataset(1000)
df.to_csv("../complaints.csv", index=False)

print("Final dataset generated: complaints.csv")

Final dataset generated: complaints.csv
